---
title: "Deploy your first managed workflow"
description: "Write, upload, and run your first managed workflow, then get its results."
---

{/* cspell:ignore writefile */}

# Deploy your first managed workflow

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-catalog~=0.21.0
qiskit[all]~=2.6.1
qiskit-serverless~=0.37.0
```
</AccordionItem>
</Accordion>

<Admonition type="caution" title="Placeholder content">
- **Client class name.** `QiskitServerless` is being renamed. Update the code examples after the new name is decided.
- **Methods to get workflows.** The current release uses `load()` to get an uploaded workflow by title. An open pull request in `qiskit-ibm-catalog` (#139) renames `load()` to `function()` and `list()` to `functions()`.
- **Runner setting.** The example sets `runner` on `QiskitFunction` because the service still uses the earlier runner when `runner` is omitted. Confirm whether users must set it, or whether the default changes before release.
- **AI skill.** A managed workflows skill is in review in the `qiskit-function-templates` repository. Update the AI skills note after it is published.
</Admonition>

This page takes you along the shortest path from an installed client to a result from the cloud. The example workflow optimizes and simulates a Qiskit circuit. It simulates the circuit on classical hardware, so it does not use QPU time.

Before you begin, [install the client package and authenticate](/docs/managed-workflows/install-managed-workflows).

{/* REVIEW: skill in review, qiskit-function-templates PR #54 */}

<Admonition type="note" title="Use AI skills (optional)">
To deploy your own code with help from an AI assistant, see [Qiskit AI skills](/docs/guides/ai-skills).
</Admonition>

## Write a workflow

### Create the folder structure

A managed workflow is a folder of Python files with one entrypoint file. The service uploads everything in the folder, up to 50 MB, and runs the entrypoint as a script. Because the service uploads the whole folder, keep your local files, such as `deploy.ipynb`, outside of it. Create the following layout:

```text
hello_world_project/
├── deploy.ipynb         # Your local notebook: upload and run the workflow
└── hello_world/         # The working directory that you upload
    └── hello_world.py   # The entrypoint
```

### Write the hello world entrypoint

The entrypoint reads a circuit, optimizes and simulates it, and saves the counts and two-qubit depth. Save the following code as `hello_world/hello_world.py`:

<Admonition type="note">
If you read this page as a notebook, the code cell for `hello_world.py` starts with the `%%writefile` [magic command](https://ipython.readthedocs.io/en/stable/interactive/magics.html#cellmagic-writefile). Running this cell saves the code to the file instead of running it.
</Admonition>

In [1]:
# This cell is hidden from users. It creates the working directory.
from pathlib import Path

Path("./hello_world").mkdir(exist_ok=True)

In [2]:
%%writefile ./hello_world/hello_world.py
from qiskit.primitives import StatevectorSampler
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_serverless import Job, get_arguments, save_result, update_status

# Read the arguments that you pass to run()
arguments = get_arguments()
circuit = arguments["circuit"]
shots = arguments.get("shots", 1024)

# Optimize the circuit. Printed text goes to the job's logs.
update_status(Job.OPTIMIZING_HARDWARE)
pass_manager = generate_preset_pass_manager(
    optimization_level=1, basis_gates=["cz", "rz", "sx", "x"]
)
isa_circuit = pass_manager.run(circuit)
two_qubit_depth = isa_circuit.depth(lambda instruction: instruction.operation.num_qubits == 2)
print(f"Two-qubit depth after optimization: {two_qubit_depth}", flush=True)

# Simulate the circuit and return the counts
sampler = StatevectorSampler(seed=1234)
pub_result = sampler.run([isa_circuit], shots=shots).result()[0]
save_result({"counts": pub_result.data.meas.get_counts(), "two_qubit_depth": two_qubit_depth})

Writing ./hello_world/hello_world.py


The entrypoint uses three functions from `qiskit_serverless`:

- `get_arguments()` returns the keyword arguments that you pass to `run()`, as a dictionary. Each value must be JSON-serializable, or an object that the client can encode, such as a `QuantumCircuit`. Encoded objects arrive as the same objects that you passed.
- `update_status(Job.OPTIMIZING_HARDWARE)` sets the detailed status `RUNNING: OPTIMIZING_FOR_HARDWARE`, which you can see while the job runs.
- `save_result()` stores a JSON-serializable value that you read later with `job.result()`. The workflow returns the counts as a dictionary and the two-qubit depth of the optimized circuit as an integer.

Anything that the workflow writes to standard output, such as the two-qubit depth, goes to the job's logs. You can also use the logger from `get_logger()`. For more about logging, see [Handle errors and report progress](/docs/managed-workflows/managed-workflow-errors-and-progress).

## Upload the workflow

With the workflow code saved, we can upload it. In your local notebook, run the following code from the `hello_world_project` folder. The code connects with your saved account, defines the workflow, and uploads it. `entrypoint` is the file that the service runs, and `working_dir` is the folder that you upload:

{/* REVIEW: class rename pending */}
{/* REVIEW: runner value. Production 0.37.0 uses the earlier runner when runner is omitted. Confirm whether this line is required. */}

In [3]:
from qiskit_ibm_catalog import QiskitFunction, QiskitServerless

serverless = QiskitServerless()

hello_world = QiskitFunction(
    title="hello-world",
    entrypoint="hello_world.py",
    working_dir="./hello_world/",
    runner="fleets",
)

hello_world = serverless.upload(hello_world)

## Run the workflow with inputs

Next, construct a three-qubit GHZ circuit to pass to `hello_world` as an input:

In [4]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.measure_all()

Run `hello_world` with the circuit and the number of shots as keyword arguments. The argument names match the keys that the entrypoint reads with `get_arguments()`:

In [5]:
job = hello_world.run(circuit=circuit, shots=1024)
print(job.job_id)

39ce7bb6-708b-4b22-9513-cb01d48e5096


`run()` returns a `Job` after the service queues the job. It does not wait for the job to run. The job continues in the cloud after you close your Python session. To find it again later, list your recent jobs with `serverless.jobs()`.

{/* REVIEW: add the web UI for job history when it is available. */}

## Get the status, result, and logs

Let's check on the job from your local notebook, starting with its status. Right after you call `run()`, the status is usually `QUEUED`:

In [6]:
job.status()

'QUEUED'

The status changes from `QUEUED` to `INITIALIZING`, `RUNNING`, and then `DONE`.

Next, get the result. `job.result()` waits until the job ends, then returns the value that the workflow passed to `save_result()`:

In [7]:
job.result()

{'counts': {'111': 514, '000': 510}, 'two_qubit_depth': 2}

In [8]:
# This cell is hidden from users. It checks the result.
assert set(_["counts"]) <= {"000", "111"}  # noqa: F821
assert sum(_["counts"].values()) == 1024  # noqa: F821

In [9]:
from qiskit.visualization import plot_histogram

plot_histogram(job.result()["counts"])

<Image src="/docs/images/managed-workflows/deploy-first-managed-workflow/extracted-outputs/b4e7a2fe-0.svg" alt="Output of the previous code cell" />

You can fetch logs in real time after a job is `RUNNING`. The logs include the line that the workflow printed:

In [10]:
print(job.logs())

/usr/local/lib64/python3.12/site-packages/qiskit/qpy/interface.py:378: UserWarning: The qiskit version used to generate the provided QPY file, 2.6.1, is newer than the current qiskit version 2.5.2. This may result in an error if the QPY file uses instructions not present in this current qiskit version
  warnings.warn(
Two-qubit depth after optimization: 2



In [11]:
# This cell is hidden from users. It checks the logs.
assert "Two-qubit depth after optimization" in job.logs()

## Next steps

<Admonition type="tip" title="Recommendations">
- [Define inputs and outputs](/docs/managed-workflows/managed-workflow-inputs-and-outputs) to learn which argument and result types a workflow supports.
- [Declare dependencies](/docs/managed-workflows/managed-workflow-dependencies) to use more Python packages in your workflow.
- [Choose a compute profile](/docs/managed-workflows/managed-workflow-compute-profiles) to run your workflow with more CPU, memory, or GPU resources.
- [Manage files across workflows](/docs/managed-workflows/manage-files-across-managed-workflows) to read and save files that persist between jobs.
- [Linking quantum jobs](/docs/managed-workflows/link-quantum-jobs) to run your circuits on a QPU from your workflow.
- [Check status, logs, and results](/docs/managed-workflows/monitor-managed-workflows) to learn all job statuses and how to debug a job, including one that ends with the `ERROR` status.
- To start from working code, deploy one of the [Function templates](/docs/managed-workflows/qiskit-function-templates), or adapt a template for your own workflow.
</Admonition>